In [ ]:
import pandas as pd
import os
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
# Parameter windowing
input_width = int(48)*7
label_width = 48
shift = label_width
total_window_size = input_width + shift
OUT_STEPS = label_width
# Definisikan irisan untuk input dan label
input_slice = slice(0, input_width)
label_start = total_window_size - label_width
labels_slice = slice(label_start, None)

from B_template import  scale_array, inverse_scale_array,  process_predictions, compute_metrics, compute_error, make_windows_autoregressive, make_windows

# data training

In [ ]:
data_training_awal= pd.read_excel('D:\\JAMALI FORECASTING\\B_train_data.xlsx', index_col=0)
data_training= data_training_awal.astype('float32').copy()
data_training

In [ ]:
data_training.describe()

In [ ]:
data_training['Suhu_PCA'], min_suhu, max_suhu = scale_array(
    data_training['Suhu_PCA'].values, 
    new_min=-1,
    new_max=1)

data_training['Holiday_Impact'], min_holiday, max_holiday = scale_array(
    data_training['Holiday_Impact'].values, 
    new_min=-1, 
    new_max=1
)

data_training['Beban'], min_beban, max_beban = scale_array(
    data_training['Beban'].values, 
    new_min=-1, 
    new_max=1)

In [ ]:
data_training.describe()

# data validation (testing)

In [ ]:
data_validation_awal= pd.read_excel('D:\\JAMALI FORECASTING\\B_test_data.xlsx', index_col=0)
data_validation= data_validation_awal.astype('float32').copy()
data_validation

In [ ]:
data_validation['Suhu_PCA'], _, _ = scale_array(
    data_validation['Suhu_PCA'].values,
    new_min=-1,
    new_max=1,
    orig_min=min_suhu,
    orig_max=max_suhu
)
data_validation['Holiday_Impact'], _, _ = scale_array(
    data_validation['Holiday_Impact'].values,
    new_min=-1,
    new_max=1,
    orig_min=min_holiday,
    orig_max=max_holiday
)
data_validation['Beban'], _, _ = scale_array(
    data_validation['Beban'].values, 
    new_min=-1, 
    new_max=1,
    orig_min=min_beban,
    orig_max=max_beban
)

In [ ]:
data_validation.describe()

# training model

In [ ]:
input_make_windows      =   data_training.to_numpy()
output_make_windows     =   data_training['Beban'].to_numpy()
x_train,    y_train     =   make_windows(input_make_windows, output_make_windows,total_window_size, input_slice, labels_slice)

In [ ]:
input_make_windows_validation   = data_validation.to_numpy()
output_make_windows_validation  = data_validation['Beban'].to_numpy()
x_val,  y_val                   = make_windows(input_make_windows_validation, output_make_windows_validation,total_window_size, input_slice, labels_slice)

In [ ]:
x_train.shape, y_train.shape, x_val.shape, y_val.shape

In [ ]:
x_train.dtype, y_train.dtype, x_val.dtype, y_val.dtype

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense, Input
import json
from tensorflow.keras import layers, mixed_precision, initializers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, History
import time

def build_model_tensor(input_lag, num_features, forecast_horizon):
    # 2. Build the Model
    model = Sequential()

    # --- THE ENCODER ---
    # This layer reads your 336 steps of 9 features. 
    # Because return_sequences=False (the default), it compresses the entire week 
    # into a single 1D context vector of 128 numbers. It learns the "story" of the week.
    model.add(Input(shape=(input_lag, num_features)))
    model.add(LSTM(units=96))

    # --- THE BRIDGE ---
    # The Decoder needs to output 48 steps, but it only has one context vector from the Encoder.
    # RepeatVector copies that single context vector 48 times to create a 3D grid again.
    model.add(RepeatVector(forecast_horizon))

    # --- THE DECODER ---
    # This layer reads the 48 repeated context vectors. 
    # Because return_sequences=True, it outputs a distinct prediction representation 
    # for every single one of the 48 future time steps, keeping the flow of time intact.
    model.add(LSTM(units=48, return_sequences=True))

    # --- THE OUTPUT ---
    # Notice we are using TimeDistributed here! 
    # Because the Decoder output is an actual sequence of 48 steps, we use TimeDistributed 
    # to apply a standard Dense(1) layer to each of the 48 steps individually. 
    # This converts the LSTM's hidden representations into your final Megawatt load numbers.
    model.add(TimeDistributed(Dense(units=1)))
    return model



In [ ]:
import tensorflow as tf
import time
import json
from tensorflow.keras.callbacks import History, ModelCheckpoint, EarlyStopping

def tensorflow_model(X_train_scaled, Y_train_scaled,
                   X_val_scaled,   Y_val_scaled,
                   learning_rate=1e-3,
                   target_MAE=0.001,
                   jumlah_epochs=50,
                   jumlah_sampel_batch=1,
                   jumlah_epoch_terbelakang=10,
                   save_best_model_path="model.h5",
                   validation_data=False,
                   load_model=None, seed_value=-1):

    global model, loss_history
    model= None
    loss_history = {}

    # ---------- callbacks ----------
    class MAEStop(tf.keras.callbacks.Callback):
        def __init__(self, thr): super().__init__(); self.thr = thr
        def on_epoch_end(self, epoch, logs=None):
            if logs and logs.get('mae', 1e9) < self.thr:
                print(f"\nMAE < {self.thr}. Stop."); self.model.stop_training = True

    class SaveEveryNEpoch(tf.keras.callbacks.Callback):
        def __init__(self, n, root): super().__init__(); self.n, self.root = n, root
        def on_epoch_end(self, epoch, logs=None):
            if (epoch + 1) % self.n == 0:
                fname = f"{self.root}_epoch{epoch+1:02d}.h5"
                self.model.save(fname)
                print(f"\n📦  Saved checkpoint: {fname}")

    class LiveLossLogger(tf.keras.callbacks.Callback):
        def __init__(self, filename, use_validation):
            super().__init__()
            self.filename = filename
            self.use_validation = use_validation
            self.live_history = {'loss': []}
            if self.use_validation:
                self.live_history['val_loss'] = []

        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            
            # Convert float32 from TensorFlow to native Python float for JSON compatibility
            current_loss = logs.get('loss')
            if current_loss is not None:
                self.live_history['loss'].append(float(current_loss))
            
            if self.use_validation:
                # Keras usually stores validation loss under 'val_loss'
                current_val_loss = logs.get('val_loss') 
                if current_val_loss is not None:
                    self.live_history['val_loss'].append(float(current_val_loss))
            
            # Open the file and overwrite it with the updated arrays
            with open(self.filename, 'w') as f:
                json.dump(self.live_history, f)

    # History callback to capture training curves
    history_cb = History()

    # ---------- model ----------
    input_width  = X_train_scaled.shape[1]   
    num_features = X_train_scaled.shape[2]   
    
    if load_model is None:
        model = build_model_tensor(input_width, num_features, Y_train_scaled.shape[1])
    else:
        model = tf.keras.models.load_model(load_model)

    model.summary()
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                  loss='mse',
                  metrics=['mse', 'mae',
                           tf.keras.metrics.MeanAbsolutePercentageError(name='mape')],
                  steps_per_execution=128)

    # ---------- build callback list ----------
    json_filename = f"loss_history_{seed_value}.json"
    
    cb = [
        history_cb,
        MAEStop(target_MAE),
        SaveEveryNEpoch(10, root=save_best_model_path.rstrip('.h5')),
        LiveLossLogger(json_filename, validation_data)
    ]

    if validation_data:
        lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_mse',    
            factor=0.5,           
            patience=4,           
            min_lr=1e-6,          
            verbose=1             
        )
        
        cb += [
            lr_scheduler,
            ModelCheckpoint(save_best_model_path,
                            monitor='val_mse', mode='min',
                            save_best_only=True, verbose=1),
            EarlyStopping(monitor='val_mse', mode='min',
                          patience=jumlah_epoch_terbelakang,
                          restore_best_weights=True, verbose=1)
        ]
    else:
        lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='mse',        
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1
        )
        
        cb += [
            lr_scheduler,
            ModelCheckpoint(save_best_model_path,
                            monitor='mse', mode='min',
                            save_best_only=True, verbose=1),
            EarlyStopping(monitor='mse', mode='min',
                          patience=jumlah_epoch_terbelakang,
                          restore_best_weights=True, verbose=1)
        ]

    # ---------- training with exception handling ----------
    start = time.time()
    try:
        model.fit(X_train_scaled, Y_train_scaled,
                  epochs=jumlah_epochs,
                  batch_size=jumlah_sampel_batch,
                  shuffle=False,
                  validation_data=(X_val_scaled, Y_val_scaled) if validation_data else None,
                  validation_batch_size=jumlah_sampel_batch,
                  callbacks=cb,
                  verbose=True)
    except Exception as e:
        print("Training stopped with error:", e)
    finally:
        # We can read the final saved state from the file to populate the return variable
        try:
            with open(json_filename, 'r') as f:
                loss_history = json.load(f)
        except FileNotFoundError:
            loss_history = {}
        print(f"Final training and validation loss securely saved to {json_filename}")

    print("Train time (s):", time.time() - start)

    # ---------- evaluate ----------
    loss, mse, mae, mape = model.evaluate(X_train_scaled, Y_train_scaled, verbose=0)
    print(f"Train  Loss={loss:.5f}  MSE={mse:.5f}  MAE={mae:.5f}  MAPE={mape:.3f}%")
    return model, loss, mse, mae, mape, loss_history

In [ ]:

seed_vektor= [20]
for seed_value in seed_vektor:
    print(f"Running with seed: {seed_value}")
    tf.random.set_seed(seed_value)
    save_best_model_path = "EDB2_%s.h5"%seed_value
    model,     loss, MSE, MAE, MAPE, loss_history  = tensorflow_model(x_train, y_train, x_val, y_val, learning_rate=0.001, target_MAE=0.001,  jumlah_epochs=100, jumlah_sampel_batch=16, 
                                                    jumlah_epoch_terbelakang=10,  save_best_model_path = save_best_model_path, 
                                                    validation_data=True, load_model=None, seed_value=seed_value)
    print(loss, MSE, MAE, MAPE)
    save_best_model_path
    model.load_weights(save_best_model_path)
    print("Loading best model from: ", save_best_model_path)
    print("model.evaluate(x_val, y_val): ", model.evaluate(x_val, y_val))
    predictions = model.predict(x_train[slice(None, None, label_width), :, :])
    predictions_reshaped = predictions.reshape(-1,)
    predictions_unscaled = inverse_scale_array(predictions_reshaped, orig_min=min_beban, orig_max=max_beban)
    df_pred_act = compute_error(x_train, data_training_awal['Beban'], ['Beban'], model, input_width,label_width, min_beban, max_beban)
    print(df_pred_act)
    df= df_pred_act.copy()
    plt.figure(figsize=(16, 6))

    # Plot the "Actual" as a green line with dot markers
    plt.plot(
        df.index, 
        df['Aktual'], 
        label='Actual output', 
        marker='.', 
        markersize=5, 
        color='#2ca02c', 
        zorder=-100
    )

    # Scatter plot for "Prediction" in orange X markers
    plt.plot(
        df.index, 
        df['Prediksi'], 
        marker='x', 
        markersize=5, 
        #edgecolors='k', 
        label='Prediction', 
        c='#ff7f0e', 
        #s=15
    )

    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.xticks(rotation=30)
    plt.title(' Actual and Predictions Over Time')
    plt.tight_layout()
    plt.show()







    df_pred_act = compute_error(x_val, data_validation_awal['Beban'], ['Beban'], model, input_width,label_width, min_beban, max_beban)
    print(df_pred_act)

    df= df_pred_act.copy()
    plt.figure(figsize=(16, 6))

    # Plot the "Actual" as a green line with dot markers
    plt.plot(
        df.index, 
        df['Aktual'], 
        label='Actual output', 
        marker='.', 
        markersize=5, 
        color='#2ca02c', 
        zorder=-100
    )

    # Scatter plot for "Prediction" in orange X markers
    plt.plot(
        df.index, 
        df['Prediksi'], 
        marker='x', 
        markersize=5, 
        #edgecolors='k', 
        label='Prediction', 
        c='#ff7f0e', 
        #s=15
    )

    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.xticks(rotation=30)
    plt.title(' Actual and Predictions Over Time')
    plt.tight_layout()
    plt.show()
